# v0.25 Risk-Aware Portfolio Ranking — Defense / Reproduction Notebook

This notebook reproduces the v0.25 allocator-development stage without using API keys. It consumes the immutable v0.24b artifact, verifies the exact persisted-model environment, reruns the validation-only ranking tournament, and keeps all PAPER/LIVE flags false.

**Important:** v0.24d OKX/KuCoin evidence is spent and may be uploaded only for post-hoc diagnostics. It cannot promote v0.25. Future-time promotion evidence starts no earlier than `2026-09-11T12:00:00Z`.

In [ ]:
!git clone --depth 1 --branch research/v25-risk-aware-ranking-forward https://github.com/parsa314/modular-crypto-trading-bot.git
%cd modular-crypto-trading-bot

In [ ]:
!python -m pip install -q -e '.[dev]'
!python -m pip install -q --upgrade --force-reinstall numpy==2.5.3 pandas==3.0.5 scikit-learn==1.9.1 joblib==1.6.0 scipy==1.18.1 threadpoolctl==3.6.0 cloudpickle==3.1.2
!python -m pip check

## Upload immutable evidence artifacts
Upload `v24b-family-portfolio-34578494059.zip`. Optionally also upload `v24d-frozen-snapshot-external-34585325516.zip` for spent-sample diagnostics. No token, password or exchange credential is required.

In [ ]:
from google.colab import files
uploaded = files.upload()
print(list(uploaded))

In [ ]:
from pathlib import Path
import zipfile
Path('frozen/v24b').mkdir(parents=True, exist_ok=True)
Path('frozen/v24d').mkdir(parents=True, exist_ok=True)
for name in uploaded:
    target = 'frozen/v24b' if 'v24b-family-portfolio' in name else ('frozen/v24d' if 'v24d-frozen-snapshot' in name else None)
    if target:
        with zipfile.ZipFile(name) as z: z.extractall(target)
print('v24b files:', sorted(p.name for p in Path('frozen/v24b').iterdir()))

In [ ]:
!pytest -q tests/test_frozen_snapshot_v24d.py tests/test_risk_ranker_v25.py

In [ ]:
import os, subprocess
cmd = ['python', 'scripts/run_v25_risk_aware_ranking.py', '--v24b-dir', 'frozen/v24b', '--output-dir', 'artifacts/v25-risk-aware-ranking']
if os.path.exists('frozen/v24d/decision.json'):
    cmd += ['--v24d-dir', 'frozen/v24d']
subprocess.run(cmd, check=True)

In [ ]:
import json, pandas as pd
decision = json.load(open('artifacts/v25-risk-aware-ranking/decision.json'))
display(pd.read_csv('artifacts/v25-risk-aware-ranking/ranker_validation_leaderboard.csv'))
decision

## Interpretation
A validation winner is only a frozen **CHALLENGER**. It is not a profitable-strategy claim and cannot authorize PAPER replacement or LIVE execution. The next scientific read must be from genuinely future-time data after the frozen boundary, followed by MTM/CVaR/correlation and search-aware statistical gates.